# Uchuu example: NFW and subhalo satellites

Load a simulation with the existing `UchuuSim(**default_uchuu_param)` API,
then pass that catalogue to `HOD`. The default is **DESIY1_DDE at z = 0.49**
from the supplied data directory.

Generate an LRG mock with analytic NFW satellite positions and Gaussian
velocities. If the loader provides subhalos, generate a second mock with the
same HOD parameters and seed and compare occupations, satellite radii and
velocities, $w_p$, and the redshift-space monopole and quadrupole.

**The existing loader disables subhalos for DDE simulations.** With the default
DESIY1_DDE selection, this notebook runs NFW and explicitly skips the subhalo
comparison. To use the loader's existing subhalo support, select `Uchuu2Gpc`
with preprocessed catalogues under `Uchuu_halo_catalogs/`. No loader methods
are changed or replaced in this notebook.

Run the cells in order with a HODDIES kernel containing NumPy, Numba,
Matplotlib, h5py, mpytools, cosmoprimo with CLASS, and pycorr with Corrfunc.
Use a compute node for the full snapshot (about 52 GiB and 239 million rows).
These parameters illustrate the workflow and are not fitted to observations.
Positions are in comoving $h^{-1}\mathrm{Mpc}$, halo radii in comoving
$h^{-1}\mathrm{kpc}$, masses in $h^{-1}M_\odot$, and velocities in km/s.

In [ ]:
from pathlib import Path
from copy import deepcopy
import os
import sys
import tempfile
import time

# Import this checkout when opened from the project or Notebooks/ directory.
repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "HODDIES" / "hod.py").is_file()), None,
)
if repo_root is not None:
    sys.path.insert(0, str(repo_root))
os.environ.setdefault(
    "NUMBA_CACHE_DIR", str(Path(tempfile.gettempdir()) / "hoddies-uchuu-example-numba")
)

import h5py
try:
    import hdf5plugin  # Registers optional HDF5 compression filters.
except ImportError:
    pass
import numpy as np
import matplotlib.pyplot as plt
from HODDIES import HOD
from HODDIES.sim_loader import setup_logging
from HODDIES.sim_loader.Uchuu_io import UchuuSim

plt.rcParams.update({"figure.dpi": 110, "font.size": 11})
NTHREADS = 4
FIX_SEED = 42

## 1. Set the simulation parameters

As in the Abacus example, put loader options in one dictionary. `path_to_sim`
is the **Uchuu root directory**: the existing loader appends
`DDE/DESIY1_DDE/out_7.rockstar.h5` for this selection. `load_particles=True`
requests subhalos; the loader determines whether that simulation supports them.

The supplied DESIY1_DDE snapshot header reports a 2000 Mpc/h box. The current
loader hard-codes 1000 for DDE, so the explicit dictionary value is applied to
the loaded catalogue below before computing densities or clustering.

In [ ]:
default_uchuu_param = {
    "boxsize": 2000.0,  # Mpc/h; verified in the supplied snapshot header.
    "path_to_sim": "/global/homes/a/arocher/users_arocher/HODDIES_data/Uchuu",
    "mass_cut": 11.5,  # Minimum log10(Mvir / (Msun/h)); None keeps all hosts.
    "z_simu": 0.49,
    "sim_name": "DESIY1_DDE",  # Also: Planck18, Planck18_DDE, Uchuu2Gpc.
    "load_particles": True,  # Request subhalos (supported by Uchuu2Gpc).
    "nthreads": NTHREADS,
}
# For preprocessed Uchuu2Gpc catalogues, change sim_name to "Uchuu2Gpc"
# and path_to_sim to the directory containing Uchuu_halo_catalogs/.

SIM_NAME = default_uchuu_param["sim_name"]
HOST_MASS_CUT = default_uchuu_param["mass_cut"]
PHASE_COLS = ("x", "y", "z", "vx", "vy", "vz")

## 2. Load the simulation once

Initialize logging, unpack the parameter dictionary, and reuse the returned
simulation object for HOD generation. No custom reader, subclass, or patch is
needed. The explicit box-size override below changes only this catalogue object.

In [ ]:
setup_logging(stream=sys.stdout)
start = time.perf_counter()
simulation = UchuuSim(**default_uchuu_param)

if default_uchuu_param["boxsize"] is not None:
    simulation.boxsize = float(default_uchuu_param["boxsize"])
BOX_SIZE = float(simulation.boxsize)
Z_SIM = simulation.z_simu
volume = BOX_SIZE**3
HAS_SUBHALOS = simulation.part_subsamples is not None and len(simulation.part_subsamples) > 0
print(f"Loaded {SIM_NAME} at z={Z_SIM}; box side: {BOX_SIZE:g} Mpc/h")
print(f"Halos: {len(simulation.hcat):,}; elapsed: {time.perf_counter() - start:.1f} s")
if HAS_SUBHALOS:
    print(f"Subhalos: {len(simulation.part_subsamples):,}")
else:
    print("No subhalos loaded: running NFW only; the subhalo comparison will be skipped.")
simulation.plot_HMF()
plt.show()

## 3. Generate the mock catalogues

`density=None` preserves the supplied HOD amplitudes. `exp_frac=0` and
`nfw_rescale=1` select the standard NFW spatial profile; `rd_normal` uses the
host velocity dispersion for analytic satellite velocities.

When subhalos are available, set `use_particles=True` for the second run.
With `f_sigv=1`, sampled subhalo velocities are preserved. The library may use
analytic placement when a host has too few subhalos. A fixed seed and thread
count hold the occupation draw constant between the two runs.

In [ ]:
LRG_PARAMETERS = dict(
    HOD_model="SHOD", Ac=1.0, log_Mcent=12.75, sigma_M=0.2,
    satellites=True, sat_HOD_model="Nsat_pow_law", As=1.0,
    M_0=12.5, M_1=13.8, alpha=1.0, nu=1.0,
    link_sat_to_central=True, conformity_bias=False, density=None,
    vel_sat="rd_normal", f_sigv=1.0, f_vcen=0.0, vsmear=0.0,
    exp_frac=0.0, exp_scale=1.0, nfw_rescale=1.0,
)
CLUSTERING_SETTINGS = {
    "rsd": True, "los": "z",
    "xi_rppi": {
        "rp_min": 0.1, "rp_max": 30.0, "n_rp_bins": 20, "pimax": 40,
    },
    "xi_smu": {
        "smin": 0.1, "smax": 40.0, "n_s_bins": 22,
        "n_mu_bins": 81, "multipole_index": [0, 2],
    },
}
hod = HOD(
    simulation, tracers=["LRG"], LRG=deepcopy(LRG_PARAMETERS),
    clustering_settings=deepcopy(CLUSTERING_SETTINGS),
    cosmo={"engine": "class"}, use_assembly_bias=False, use_particles=False,
    nthreads=NTHREADS, seed=FIX_SEED,
)
if hod.cosmo is None:
    raise RuntimeError("The Uchuu cosmology is required for redshift-space clustering.")

mocks = {}
hod.args["use_particles"] = False
mocks["NFW"] = hod.make_mock_cat(tracers="LRG", fix_seed=FIX_SEED)
if HAS_SUBHALOS:
    hod.args["use_particles"] = True
    mocks["Subhalos"] = hod.make_mock_cat(tracers="LRG", fix_seed=FIX_SEED)

# Use canonical periodic coordinates in double precision for pair counting.
for mock in mocks.values():
    for coordinate in ("x", "y", "z"):
        mock[coordinate] = np.remainder(mock[coordinate].astype(np.float64), BOX_SIZE)

if HAS_SUBHALOS:
    nfw, subhalos = mocks["NFW"], mocks["Subhalos"]
    np.testing.assert_array_equal(nfw["halo_id"], subhalos["halo_id"])
    np.testing.assert_array_equal(nfw["Central"], subhalos["Central"])
    central = nfw["Central"] == 1
    for column in PHASE_COLS:
        np.testing.assert_array_equal(nfw[column][central], subhalos[column][central])

print(f"{'Prescription':<16} {'N galaxies':>12} {'n [h^3/Mpc^3]':>17} {'f_sat':>9}")
for name, mock in mocks.items():
    print(f"{name:<16} {len(mock):>12,} {len(mock)/volume:>17.4e} "
          f"{np.mean(mock['Central'] == 0):>9.3f}")

## 4. Plot occupations and satellite phase space

Divide galaxy counts by **all input halos** in each mass bin, including empty
hosts. For phase space, match each satellite to its host ID and use the minimum
periodic displacement. Convert host `Rh` from kpc/h to Mpc/h when forming
$r/R_{\rm vir}$. If both mocks exist, their measured occupations should coincide.

In [ ]:
mass_min = HOST_MASS_CUT if HOST_MASS_CUT is not None else float(simulation.hcat["log10_Mh"].min())
mass_edges = np.linspace(mass_min, 15.5, 30)
mass_centres = 0.5 * (mass_edges[1:] + mass_edges[:-1])
halo_counts = np.histogram(simulation.hcat["log10_Mh"], mass_edges)[0]
fig, ax = plt.subplots(figsize=(7, 4))
for name, mock in mocks.items():
    for flag, kind in [(1, "centrals"), (0, "satellites")]:
        counts = np.histogram(mock["log10_Mh"][mock["Central"] == flag], mass_edges)[0]
        occupation = np.divide(counts, halo_counts, out=np.full(counts.shape, np.nan), where=halo_counts > 0)
        ax.plot(mass_centres, occupation, marker="o" if name == "NFW" else None,
                ls="-" if name == "NFW" else "--", label=f"{name}: {kind}")
ax.set(xlabel=r"$\log_{10}(M_h/[h^{-1}M_\odot])$", ylabel=r"$\langle N|M_h\rangle$", yscale="log")
ax.legend(fontsize=8)
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

host_order = np.argsort(simulation.hcat["halo_id"])
sorted_host_ids = simulation.hcat["halo_id"][host_order]
phase_space = {}
for name, mock in mocks.items():
    satellites = mock[mock["Central"] == 0]
    host_index = host_order[np.searchsorted(sorted_host_ids, satellites["halo_id"])]
    radius2 = np.zeros(len(satellites), dtype=np.float64)
    for coord in ("x", "y", "z"):
        delta = satellites[coord] - simulation.hcat[coord][host_index]
        delta = (delta + BOX_SIZE / 2) % BOX_SIZE - BOX_SIZE / 2
        radius2 += delta.astype(np.float64)**2
    relative_radius = np.sqrt(radius2) / (simulation.hcat["Rh"][host_index] / 1000.0)
    dvz = satellites["vz"] - simulation.hcat["vz"][host_index]
    phase_space[name] = (relative_radius, dvz)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, (radius, dvz) in phase_space.items():
    positive = radius > 0
    axes[0].hist(np.log10(radius[positive]), bins=np.linspace(-3, 1.5, 60),
                 density=True, histtype="step", label=name)
    axes[1].hist(dvz, bins=np.linspace(-2000, 2000, 80),
                 density=True, histtype="step", label=name)
axes[0].set(xlabel=r"$\log_{10}(r/R_{\rm vir})$", ylabel="PDF within plotted range")
axes[1].set(xlabel=r"$v_{{\rm sat},z}-v_{h,z}\;[\mathrm{km\,s^{-1}}]$", ylabel="PDF within plotted range")
for ax in axes:
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

## 5. Plot projected and redshift-space clustering

Measure $w_p$ with $\pi_{\max}=40$ Mpc/h, $\xi_0$, and $\xi_2$ in redshift space
along $z$. The same bins and geometric bin centres are used for every mock.
Full-volume pair counts can take time.

If a subhalo mock exists, lower panels show its fractional difference in $w_p$
and its differences in $s^2\xi_0$ and $s^2\xi_2$ relative to NFW. The quadrupole
can cross zero, so an absolute difference is used. A single pair of mocks does
not provide uncertainty estimates. When subhalos are unavailable, plot only the
NFW statistics.

In [ ]:
pair = "LRG_LRG"
statistics = {}
for name, mock in mocks.items():
    start = time.perf_counter()
    statistics[name] = hod.compute_stats(mock, stat=["wp", "xi_ells"], tracers=["LRG"], verbose=True)
    for statistic, section, edge_key in [("wp", "xi_rppi", "edges_rppi"),
                                        ("xi_ells", "xi_smu", "edges_smu")]:
        edges = hod.args["clustering_settings"][section][edge_key][0]
        centres = np.sqrt(edges[:-1] * edges[1:])
        statistics[name][statistic][pair] = (centres, statistics[name][statistic][pair][1])
    print(f"{name}: clustering measured in {time.perf_counter() - start:.1f} s")

nrows = 2 if HAS_SUBHALOS else 1
fig, axes = plt.subplots(nrows, 3, figsize=(15, 3.5 * nrows), sharex="col", squeeze=False)
for name, result in statistics.items():
    rp, wp = result["wp"][pair]
    s, xi = result["xi_ells"][pair]
    axes[0, 0].semilogx(rp, rp * wp, label=name)
    for index, ell in enumerate([0, 2]):
        axes[0, index + 1].semilogx(s, s**2 * xi[index], label=name)

axes[0, 0].set_ylabel(r"$r_p w_p\;[h^{-2}\mathrm{Mpc}^2]$")
for index, ell in enumerate([0, 2]):
    axes[0, index + 1].set_ylabel(rf"$s^2\xi_{ell}\;[h^{{-2}}\mathrm{{Mpc}}^2]$")
if HAS_SUBHALOS:
    rp, wp_nfw = statistics["NFW"]["wp"][pair]
    rp_sub, wp_sub = statistics["Subhalos"]["wp"][pair]
    s, xi_nfw = statistics["NFW"]["xi_ells"][pair]
    s_sub, xi_sub = statistics["Subhalos"]["xi_ells"][pair]
    np.testing.assert_allclose(rp, rp_sub, equal_nan=True)
    np.testing.assert_allclose(s, s_sub, equal_nan=True)
    valid = np.isfinite(wp_nfw) & np.isfinite(wp_sub) & (wp_nfw != 0)
    axes[1, 0].semilogx(rp[valid], (wp_sub[valid] - wp_nfw[valid]) / wp_nfw[valid])
    axes[1, 0].set_ylabel(r"$\Delta w_p/w_p^{\rm NFW}$")
    for index, ell in enumerate([0, 2]):
        axes[1, index + 1].semilogx(s, s**2 * (xi_sub[index] - xi_nfw[index]))
        axes[1, index + 1].set_ylabel(rf"$s^2\Delta\xi_{ell}\;[h^{{-2}}\mathrm{{Mpc}}^2]$")
    for ax in axes[1]:
        ax.axhline(0, color="0.5", lw=0.8)
axes[-1, 0].set_xlabel(r"$r_p\;[h^{-1}\mathrm{Mpc}]$")
for ax in axes[-1, 1:]:
    ax.set_xlabel(r"$s\;[h^{-1}\mathrm{Mpc}]$")
for ax in axes[0]:
    ax.legend(fontsize=8)
for ax in axes.flat:
    ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

## 6. Optional: save results

The available mocks remain in `mocks`, and their clustering measurements in
`statistics`. Set the flag below to save them with the run parameters. The
simulation files are only read. Repeating this cell replaces output files for
the same seed.

In [ ]:
SAVE_RESULTS = False
if SAVE_RESULTS:
    import json
    output_dir = (repo_root / "Notebooks" if repo_root is not None else Path.cwd()) / "outputs" / "uchuu_example"
    output_dir.mkdir(parents=True, exist_ok=True)
    columns = [*PHASE_COLS, "halo_id", "log10_Mh", "Central", "TRACER"]
    arrays = {}
    for name, mock in mocks.items():
        slug = name.lower()
        np.savez_compressed(output_dir / f"{slug}_seed{FIX_SEED}.npz",
                            **{column: mock[column] for column in columns})
        rp, wp = statistics[name]["wp"][pair]
        s, xi = statistics[name]["xi_ells"][pair]
        arrays.update({f"{slug}_rp": rp, f"{slug}_wp": wp,
                       f"{slug}_s": s, f"{slug}_xi_ells": xi})
    np.savez_compressed(output_dir / f"clustering_seed{FIX_SEED}.npz", **arrays)
    run_parameters = dict(
        simulation=default_uchuu_param, boxsize=BOX_SIZE,
        seed=FIX_SEED, nthreads=hod.nthreads, has_subhalos=HAS_SUBHALOS,
        hod=LRG_PARAMETERS, clustering=CLUSTERING_SETTINGS,
    )
    (output_dir / f"parameters_seed{FIX_SEED}.json").write_text(json.dumps(run_parameters, indent=2))
    print("Saved results to", output_dir)